# Quick Links Bank for CodeSignal Screen

## Core Math & einsum
* `numpy.einsum` (The Holy Grail): https://numpy.org/doc/stable/reference/generated/numpy.einsum.html
* `numpy.inner` / `numpy.outer`: https://numpy.org/doc/stable/reference/generated/numpy.outer.html
* `numpy.trace`: https://numpy.org/doc/stable/reference/generated/numpy.trace.html

## Linear Algebra (SVD & Eig)
* `numpy.linalg.svd`: https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html
* `numpy.linalg.eigh` (Symmetric): https://numpy.org/doc/stable/reference/generated/numpy.linalg.eigh.html
* `numpy.linalg.pinv` (Pseudo-inverse): https://numpy.org/doc/stable/reference/generated/numpy.linalg.pinv.html

## Masking & Matrix Manipulation
* `numpy.triu` (Upper triangle / Causal mask): https://numpy.org/doc/stable/reference/generated/numpy.triu.html
* `numpy.tril` (Lower triangle): https://numpy.org/doc/stable/reference/generated/numpy.tril.html
* `numpy.diag`: https://numpy.org/doc/stable/reference/generated/numpy.diag.html

## Numerical Stability & Testing
* `numpy.allclose` (Float comparison): https://numpy.org/doc/stable/reference/generated/numpy.allclose.html
* `numpy.max` (For LogSumExp shift): https://numpy.org/doc/stable/reference/generated/numpy.max.html
* `numpy.exp` / `numpy.log`: https://numpy.org/doc/stable/reference/generated/numpy.exp.html

# Addendum: Core Elhage 2021 Concepts
*Keep with Memorize Sheet*

## 1. The Path Expansion Trick
Attention-only models can be written as a sum of interpretable end-to-end functions mapping tokens to changes in logits. 
- **0-Layer (Direct) Path:** The model's baseline bigram statistics. Embeddings multiply directly with unembeddings.
  - $\text{Logits}_{\text{direct}} = W_E W_U$
- **1-Layer Path:** The sum of the effects of every individual attention head.
  - $\text{Logits}_{\text{1-layer}} = \sum_{h} (W_E W_{OV}^{h} W_U)$
- **Total Logits:** $\text{Logits} = W_E W_U + \sum_{h} (W_E W_{OV}^{h} W_U)$

## 2. QK vs. OV Asymmetry
Attention heads contain two largely independent computations:
- **QK Circuit ($W_{QK} = W_Q W_K^T$):** Dictates *where* to move information. It calculates the attention pattern (the weights).
- **OV Circuit ($W_{OV} = W_V W_O$):** Dictates *what* information to move. It governs which subspace of the residual stream the head reads from, and which subspace it writes to. 
- *Interpretability Rule of Thumb:* If you are looking at attention heatmaps, you are analyzing the QK circuit. If you are doing SVD to see what features a head represents, you are analyzing the OV circuit.

You're completely right. Let's make this a single, unified block of Markdown with zero conversational filler inside it, and we will use proper LaTeX for the math so there is absolute clarity on the calculus before you write the code.


# Emergency Backprop & Attribution Reference

## 1. Variable Mapping

| Math Derivative | Code Variable | Shape | Description |
| :--- | :--- | :--- | :--- |
| $\frac{\partial L}{\partial O}$ | `gO` | `(b, i, m)` | Gradient of Loss wrt Attention Output |
| $\frac{\partial L}{\partial A}$ | `gA` | `(b, h, i, j)` | Gradient of Loss wrt Attention Probabilities |
| $\frac{\partial L}{\partial S}$ | `gS` | `(b, h, i, j)` | Gradient of Loss wrt Raw Attention Scores |
| $\frac{\partial L}{\partial Q}$ | `gQ` | `(b, h, i, a)` | Gradient of Loss wrt Queries |

---

## 2. Phase 1: Through the Value Matrix
To find how the probabilities $A$ affected the output $O = AV$:

**Math:**
$$\frac{\partial L}{\partial A} = \frac{\partial L}{\partial O} V^T$$

**Code (NumPy):**
```python
# Assuming gO has been projected back through W_O into head-space (b, h, i, a)
# V is (b, h, j, a)
gA = np.einsum('bhia, bhja -> bhij', gO, V)

```

## 3. Phase 2: The Softmax Jacobian

To find how the raw scores $S$ affected the probabilities $A$, accounting for the sum-to-one constraint across the source/key dimension $j$.

**Math:**


$$ \frac{\partial L}{\partial S_{ij}} = A_{ij} \left( \frac{\partial L}{\partial A_{ij}} - \sum_{k} A_{ik} \frac{\partial L}{\partial A_{ik}} \right) $$

**Code (NumPy):**

```python
# Sum weighted gradients across the SOURCE (j) dimension
weighted_sum = np.sum(A * gA, axis=-1, keepdims=True)
gS = A * (gA - weighted_sum)

```

## 4. Phase 3: Through the Query-Key Dot Product

To find how the Queries and Keys affected the raw scores $S = \frac{Q K^T}{\sqrt{d_k}}$.

**Math:**


$$ \frac{\partial L}{\partial Q} = \frac{1}{\sqrt{d_k}} \frac{\partial L}{\partial S} K $$

$$ \frac{\partial L}{\partial K} = \frac{1}{\sqrt{d_k}} \left( \frac{\partial L}{\partial S} \right)^T Q $$

**Code (NumPy):**

```python
# dk is the head dimension (alpha)
scale = 1.0 / np.sqrt(dk)
gQ = scale * np.einsum('bhij, bhja -> bhia', gS, K)
gK = scale * np.einsum('bhij, bhia -> bhja', gS, Q)

```

## 5. Phase 4: To the Weight Matrices

To find how the weight matrices affected the Queries (e.g., $Q = X W_Q$).

**Math:**


$$ \frac{\partial L}{\partial W_Q} = X^T \frac{\partial L}{\partial Q} $$

**Code (NumPy):**

```python
# X is (b, i, m), gQ is (b, h, i, a)
# Summing over batch (b) and sequence position (i)
gWq = np.einsum('bim, bhia -> hma', X, gQ)

```

```

Copy that block directly. Now that the math is rigorously defined alongside your `einsum` index notation, you have a complete map to survive any gradient question they throw at you.

Are you ready to start your 120-minute timer for Practice Test 1, or is there one last edge case you want to lock down?

```